In [1]:
import sys, json, logging, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
CONFIG = {
    'SEED': 42, 'TEST_SIZE': 0.2, 'CALIBRATION_SIZE': 0.1,
    'CATBOOST_ITERATIONS': 500, 'CATBOOST_LEARNING_RATE': 0.05, 'CATBOOST_DEPTH': 6,
    'MIN_POSITIVES': 30
}

In [3]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

In [4]:
# Load numeric-only features
X_list = []
for name in best_combo:
    df = pd.read_parquet(DATA_DIR / f"features_{name}.parquet").drop(columns=['organism'], errors='ignore')
    X_list.append(df)
X_full = pd.concat(X_list, axis=1)

# Save feature column order
with open(DATA_DIR / 'train_feature_columns.json', 'w') as f:
    json.dump(X_full.columns.tolist(), f)

groups = labeled_pd['organism']
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['SEED'])
train_idx, test_idx = next(gss.split(X_full, labeled_pd[active_stressors[0]], groups=groups))

# Save split
import json
with open(DATA_DIR / 'train_test_indices.json', 'w') as f:
    json.dump({'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist()}, f)

In [ ]:
def train_stressor(stressor):
    """Train CatBoost model for a single stressor with balanced class weights."""
    y = labeled_pd[stressor]
    if y.sum() < CONFIG['MIN_POSITIVES']:
        return None
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    g_tr = groups.iloc[train_idx]
    gss_cal = GroupShuffleSplit(n_splits=1, test_size=CONFIG['CALIBRATION_SIZE'],
                                random_state=CONFIG['SEED'])
    sub_idx, cal_idx = next(gss_cal.split(X_tr, y_tr, groups=g_tr))
    X_sub, X_cal = X_tr.iloc[sub_idx], X_tr.iloc[cal_idx]
    y_sub, y_cal = y_tr.iloc[sub_idx], y_tr.iloc[cal_idx]

    # Skip if the training data has only one class
    if y_sub.nunique() < 2:
        log.warning(f"{stressor}: training set has only one class; skipping.")
        return None

    # Train CatBoost with balanced class weights to address class imbalance
    model = CatBoostClassifier(iterations=CONFIG['CATBOOST_ITERATIONS'],
                               learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
                               depth=CONFIG['CATBOOST_DEPTH'],
                               cat_features=None, eval_metric='AUC',
                               auto_class_weights='Balanced',
                               random_seed=CONFIG['SEED'], verbose=50)
    model.fit(X_sub, y_sub, verbose=50)
    
    # Get probability predictions on test set
    y_proba = model.predict_proba(X_te)[:, 1]
    
    # Use default threshold for now (will be tuned in next cell)
    y_pred = (y_proba >= 0.5).astype(int)

    # Platt calibration with single-class fallback
    cal_proba = model.predict_proba(X_cal)[:, 1]
    if y_cal.nunique() < 2:
        log.warning(f"{stressor}: calibration set has only one class; skipping Platt.")
        y_proba_cal = y_proba
        platt = None
    else:
        platt = LogisticRegression()
        platt.fit(cal_proba.reshape(-1, 1), y_cal)
        y_proba_cal = platt.predict_proba(y_proba.reshape(-1, 1))[:, 1]

    metrics = {
        'AUC': roc_auc_score(y_te, y_proba),
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_te, y_pred),
        'Pos rate': y_te.mean()
    }
    
    # Save model and calibrator
    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_final.cbm"))
    if platt is not None:
        with open(MODEL_DIR / f"stressor_{stressor}_platt.pkl", 'wb') as f:
            pickle.dump(platt, f)
    
    # Save predictions for threshold tuning
    pd.DataFrame({'y_test': y_te.values, 'y_proba': y_proba,
                  'y_proba_cal': y_proba_cal, 'group': groups.iloc[test_idx].values}
                ).to_parquet(MODEL_DIR / f"stressor_{stressor}_predictions.parquet")
    
    return metrics

In [8]:
all_metrics = {}
for s in active_stressors:
    log.info(f"Training {s}")
    m = train_stressor(s)
    if m: all_metrics[s] = m

pd.DataFrame(all_metrics).T.to_csv(DATA_DIR / 'final_model_performance.csv')
log.info("Training complete.")

2026-06-28 19:12:48,482 INFO Training Zn


0:	total: 70.4ms	remaining: 35.1s


50:	total: 960ms	remaining: 8.45s


100:	total: 1.83s	remaining: 7.21s


150:	total: 2.73s	remaining: 6.31s


...

450:	total: 7.59s	remaining: 825ms


2026-06-28 19:19:54,751 INFO Training Biofilm


499:	total: 8.3s	remaining: 0us


2026-06-28 19:19:55,067 WARNING Biofilm: training set has only one class; skipping.


2026-06-28 19:19:55,083 INFO Training complete.


In [ ]:
# Tune decision thresholds per stressor to maximize F1 score
best_thresholds = {}

for stressor in active_stressors:
    pred_file = MODEL_DIR / f"stressor_{stressor}_predictions.parquet"
    if not pred_file.exists():
        continue
    
    preds_df = pd.read_parquet(pred_file)
    y_test = preds_df['y_test'].values
    y_proba = preds_df['y_proba'].values
    
    # Sweep thresholds from 0.01 to 0.99
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_scores = []
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        f1_scores.append(f1)
    
    # Find threshold maximizing F1
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    best_thresholds[stressor] = float(best_threshold)
    log.info(f"{stressor}: best threshold={best_threshold:.3f}, F1={best_f1:.4f}")

# Save best thresholds
with open(DATA_DIR / 'best_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)

log.info(f"Saved best thresholds for {len(best_thresholds)} stressors to data/best_thresholds.json")